# Notebook 08 — Transformers Baselines and Batching

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/08_transformers_baselines_and_batching.ipynb)

Optimized runtimes are easier to understand when you first build a plain baseline. This notebook uses raw `transformers` patterns, lightweight simulations, and an optional tiny live model to show what batching actually changes.

## What you will build

- tokenize a realistic request set and inspect prompt length spread
- measure a sequential baseline
- simulate fixed-size and dynamic batching policies
- optionally run a tiny open-source model for a live smoke test
- summarize why dedicated serving runtimes outperform the raw baseline

## Practical note

The tokenizer setup is real. Model execution is optional. That keeps the notebook open-source and runnable in light environments while still showing the mechanics behind batching.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=str(CACHE_DIR))
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("✓ Tokenizer ready:", MODEL_NAME)

In [ ]:
from transformers import AutoModelForCausalLM
import torch

random.seed(8)
ARTIFACT_DIR = ARTIFACT_ROOT / "08_transformers_baselines_and_batching"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 1: Build a small request set

We keep the tasks varied because batching gains depend heavily on prompt length and output length distribution.

In [ ]:
requests = [
    {"id": "r1", "prompt": "Summarize why GGUF is useful for local inference.", "target_tokens": 90, "arrival_ms": 0},
    {"id": "r2", "prompt": "List three reasons to prefer Q4_K_M on a laptop.", "target_tokens": 80, "arrival_ms": 6},
    {"id": "r3", "prompt": "Explain the trade-off between prompt length and prefill latency.", "target_tokens": 110, "arrival_ms": 14},
    {"id": "r4", "prompt": "Give a short deployment checklist for a local OpenAI-compatible server.", "target_tokens": 120, "arrival_ms": 22},
    {"id": "r5", "prompt": "Why does dynamic batching help throughput under bursty traffic?", "target_tokens": 100, "arrival_ms": 29},
    {"id": "r6", "prompt": "Compare CPU-only inference with partial GPU offload in two paragraphs.", "target_tokens": 150, "arrival_ms": 37},
    {"id": "r7", "prompt": "Generate a JSON object with fields model, quant, and estimated_memory_gib.", "target_tokens": 70, "arrival_ms": 44},
    {"id": "r8", "prompt": "Describe why raw transformers code is a useful baseline even when faster runtimes exist.", "target_tokens": 130, "arrival_ms": 53},
]

requests_df = pd.DataFrame(requests)
requests_df

## Step 2: Inspect prompt token lengths

Prompt length spread drives prefill cost and makes naive batching awkward.

In [ ]:
tokenized_prompts = tokenizer(list(requests_df["prompt"]), add_special_tokens=True)
requests_df["prompt_tokens"] = [len(ids) for ids in tokenized_prompts["input_ids"]]
requests_df[["id", "prompt_tokens", "target_tokens", "arrival_ms"]]

## Step 3: Pad a small batch manually

This is the first mechanical cost of raw batching: you must manage padding and masks yourself or trust utilities that do it for you.

In [ ]:
mini_batch = list(requests_df["prompt"][:4])
padded = tokenizer(mini_batch, padding=True, return_tensors="pt")
manual_batch_view = pd.DataFrame({
    "shape_component": ["input_ids", "attention_mask"],
    "shape": [tuple(padded["input_ids"].shape), tuple(padded["attention_mask"].shape)],
})
manual_batch_view

## Step 4: Build a sequential baseline simulator

The simulator separates prefill and decode. That makes later batching gains easier to reason about.

In [ ]:
def simulate_sequential(record, prefill_tps=2600, decode_tps=38, overhead_ms=18):
    prefill_ms = 1000 * record["prompt_tokens"] / prefill_tps
    decode_ms = 1000 * record["target_tokens"] / decode_tps
    total_ms = prefill_ms + decode_ms + overhead_ms
    return {
        "prefill_ms": round(prefill_ms, 1),
        "decode_ms": round(decode_ms, 1),
        "latency_ms": round(total_ms, 1),
        "throughput_tps": round(record["target_tokens"] / max(total_ms / 1000, 0.001), 1),
    }

sequential_rows = []
for record in requests_df.to_dict("records"):
    sequential_rows.append({**record, **simulate_sequential(record)})

sequential_df = pd.DataFrame(sequential_rows)
sequential_df[["id", "prompt_tokens", "target_tokens", "latency_ms", "throughput_tps"]]

## Step 5: Summarize the sequential baseline

This is the number to beat. Any batching policy should improve throughput without making latency unacceptable.

In [ ]:
sequential_summary = pd.DataFrame([{
    "policy": "sequential",
    "mean_latency_ms": round(sequential_df["latency_ms"].mean(), 1),
    "p95_latency_ms": round(sequential_df["latency_ms"].quantile(0.95), 1),
    "mean_throughput_tps": round(sequential_df["throughput_tps"].mean(), 1),
}])
sequential_summary

## Step 6: Simulate fixed-size batching

Fixed batches are easy to implement but force short prompts to wait for long prompts.

In [ ]:
def simulate_fixed_batches(records, batch_size=4, prefill_tps=5400, decode_tps=70, overhead_ms=28):
    outputs = []
    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        max_prompt = max(item["prompt_tokens"] for item in batch)
        max_output = max(item["target_tokens"] for item in batch)
        batch_ms = 1000 * max_prompt / prefill_tps + 1000 * max_output / decode_tps + overhead_ms
        for item in batch:
            outputs.append({
                **item,
                "batch_id": start // batch_size,
                "latency_ms": round(batch_ms, 1),
                "throughput_tps": round(item["target_tokens"] / max(batch_ms / 1000, 0.001), 1),
            })
    return pd.DataFrame(outputs)

fixed_df = simulate_fixed_batches(requests_df.to_dict("records"), batch_size=4)
fixed_df[["id", "batch_id", "latency_ms", "throughput_tps"]]

## Step 7: Compare sequential and fixed batching

Throughput usually rises, but tail latency can become uneven. That is why smarter schedulers exist.

In [ ]:
fixed_summary = pd.DataFrame([{
    "policy": "fixed_batch_4",
    "mean_latency_ms": round(fixed_df["latency_ms"].mean(), 1),
    "p95_latency_ms": round(fixed_df["latency_ms"].quantile(0.95), 1),
    "mean_throughput_tps": round(fixed_df["throughput_tps"].mean(), 1),
}])
pd.concat([sequential_summary, fixed_summary], ignore_index=True)

## Step 8: Simulate a simple dynamic batching scheduler

Dynamic batching groups requests that arrive close together. That usually keeps throughput gains while reducing the worst waiting behavior.

In [ ]:
def simulate_dynamic_batches(records, max_batch_size=4, max_wait_ms=12, prefill_tps=6200, decode_tps=78, overhead_ms=24):
    ordered = sorted(records, key=lambda item: item["arrival_ms"])
    outputs = []
    queue = []
    dispatch_clock = 0.0
    batch_id = 0

    def flush_queue(queue, ready_time, batch_id):
        max_prompt = max(item["prompt_tokens"] for item in queue)
        max_output = max(item["target_tokens"] for item in queue)
        service_ms = 1000 * max_prompt / prefill_tps + 1000 * max_output / decode_tps + overhead_ms
        batch_finish = ready_time + service_ms
        batch_outputs = []
        for item in queue:
            batch_outputs.append({
                **item,
                "batch_id": batch_id,
                "dispatch_ms": round(ready_time, 1),
                "latency_ms": round(batch_finish - item["arrival_ms"], 1),
                "throughput_tps": round(item["target_tokens"] / max((batch_finish - item["arrival_ms"]) / 1000, 0.001), 1),
            })
        return batch_outputs, batch_finish

    for item in ordered:
        if not queue:
            queue.append(item)
            dispatch_clock = max(dispatch_clock, item["arrival_ms"])
            continue
        first_arrival = queue[0]["arrival_ms"]
        wait_time = item["arrival_ms"] - first_arrival
        if len(queue) >= max_batch_size or wait_time > max_wait_ms:
            batch_outputs, dispatch_clock = flush_queue(queue, max(dispatch_clock, first_arrival + min(wait_time, max_wait_ms)), batch_id)
            outputs.extend(batch_outputs)
            batch_id += 1
            queue = [item]
            dispatch_clock = max(dispatch_clock, item["arrival_ms"])
        else:
            queue.append(item)

    if queue:
        batch_outputs, dispatch_clock = flush_queue(queue, max(dispatch_clock, queue[0]["arrival_ms"]), batch_id)
        outputs.extend(batch_outputs)

    return pd.DataFrame(outputs)

dynamic_df = simulate_dynamic_batches(requests_df.to_dict("records"))
dynamic_df[["id", "batch_id", "dispatch_ms", "latency_ms", "throughput_tps"]]

## Step 9: Compare all three policies

The point is not to crown one universal winner. The point is to see why serving runtimes spend so much engineering effort on scheduling.

In [ ]:
dynamic_summary = pd.DataFrame([{
    "policy": "dynamic_batching",
    "mean_latency_ms": round(dynamic_df["latency_ms"].mean(), 1),
    "p95_latency_ms": round(dynamic_df["latency_ms"].quantile(0.95), 1),
    "mean_throughput_tps": round(dynamic_df["throughput_tps"].mean(), 1),
}])
policy_summary = pd.concat([sequential_summary, fixed_summary, dynamic_summary], ignore_index=True)
policy_summary

## Step 10: Plot the policy comparison

A visual summary makes the latency and throughput trade-off easier to explain to stakeholders.

In [ ]:
ax = policy_summary.plot(x="policy", y=["mean_latency_ms", "p95_latency_ms"], kind="bar", figsize=(8, 4), rot=0, title="Latency by batching policy")
ax.set_ylabel("milliseconds")
plt.tight_layout()

## Step 11: Sweep batcher settings

Real systems tune batch size and wait time together. A small sweep already shows how sensitive outcomes can be.

In [ ]:
sweep_rows = []
for batch_size in [2, 4, 6]:
    for wait_ms in [4, 8, 12, 20]:
        sweep_df = simulate_dynamic_batches(requests_df.to_dict("records"), max_batch_size=batch_size, max_wait_ms=wait_ms)
        sweep_rows.append({
            "max_batch_size": batch_size,
            "max_wait_ms": wait_ms,
            "mean_latency_ms": round(sweep_df["latency_ms"].mean(), 1),
            "mean_throughput_tps": round(sweep_df["throughput_tps"].mean(), 1),
        })

sweep_summary = pd.DataFrame(sweep_rows)
sweep_summary

## Step 12: Estimate memory pressure from larger batches

Batching improves utilization, but memory grows too. The helper below gives a rough planning estimate.

In [ ]:
def estimate_batch_memory_gib(batch_size, prompt_tokens, hidden_size=4096, layers=32, bytes_per_value=2):
    activation_bytes = batch_size * prompt_tokens * hidden_size * bytes_per_value
    kv_bytes = batch_size * prompt_tokens * hidden_size * layers * 2 * bytes_per_value
    return {
        "activation_gib_est": round(activation_bytes / (1024 ** 3), 3),
        "kv_gib_est": round(kv_bytes / (1024 ** 3), 3),
    }

memory_rows = []
for batch_size in [1, 2, 4, 8]:
    stats = estimate_batch_memory_gib(batch_size=batch_size, prompt_tokens=1024)
    memory_rows.append({"batch_size": batch_size, **stats})

memory_df = pd.DataFrame(memory_rows)
memory_df

## Step 13: Optional live model smoke test

This cell is disabled by default. When you have room, load a tiny open-source model and verify the baseline with one real generation call.

In [ ]:
LOAD_REAL_MODEL = False
REAL_MODEL_NAME = "sshleifer/tiny-gpt2"
real_tokenizer = None
real_model = None

if LOAD_REAL_MODEL:
    real_tokenizer = AutoTokenizer.from_pretrained(REAL_MODEL_NAME, cache_dir=str(CACHE_DIR))
    if real_tokenizer.pad_token_id is None:
        real_tokenizer.pad_token_id = real_tokenizer.eos_token_id
    real_model = AutoModelForCausalLM.from_pretrained(REAL_MODEL_NAME, cache_dir=str(CACHE_DIR))
    real_model.eval()
    print("Loaded", REAL_MODEL_NAME)
else:
    print("Set LOAD_REAL_MODEL = True to run a tiny live transformers baseline.")

## Step 14: Optional real generation call

The point of this cell is validation, not benchmarking greatness. Tiny models are only a sanity check for the notebook mechanics.

In [ ]:
if LOAD_REAL_MODEL:
    prompt = "Explain dynamic batching in one short paragraph."
    encoded = real_tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output_ids = real_model.generate(**encoded, max_new_tokens=60, do_sample=False)
    generated = real_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(generated)
else:
    print("Skipped live generation.")

## Step 15: Build an async request queue sketch

A raw Python service can batch requests too. The point is that you must implement the queueing logic yourself.

In [ ]:
async def collect_batch(records, max_batch_size=4):
    queue = deque(records)
    batches = []
    while queue:
        batch = []
        while queue and len(batch) < max_batch_size:
            batch.append(queue.popleft())
        batches.append(batch)
    return batches

batched_preview = asyncio.run(collect_batch(requests_df.to_dict("records"), max_batch_size=3))
[len(batch) for batch in batched_preview]

## Step 16: Explain why optimized runtimes exist

By now the pressure points are visible:

- raw `transformers` leaves scheduling to you
- padding waste grows as prompt lengths diverge
- memory accounting gets harder as batches grow
- latency and throughput goals conflict under bursty traffic

In [ ]:
artifact = {
    "notebook": "08_transformers_baselines_and_batching",
    "policy_summary": policy_summary.to_dict("records"),
    "sweep_summary": sweep_summary.to_dict("records"),
    "memory_summary": memory_df.to_dict("records"),
}

artifact_path = ARTIFACT_DIR / "transformers_batching_baseline.json"
artifact_path.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
print("Wrote artifact:", artifact_path.resolve())

## Recap

A raw `transformers` baseline is valuable because it exposes the real work: tokenization, padding, queues, batch assembly, and memory trade-offs. Once those mechanics are clear, optimized runtimes like `llama.cpp`, vLLM, and SGLang stop feeling magical and start feeling engineered.